In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Carga de datos

In [2]:
df_test = pd.read_csv(r"C:\Users\javier.sdiaz\Proyecto ML_2025\data\raw\Test_Temp_int.csv", sep=";")
df_test.head()

,Fecha_Hora,Temp_amb,Radiacion_1,Radiacion_2,Dir_viento,Vel_viento,Energia,Potencia,Oil_Temp,cosPhi,...,Idc3,Idc4,Idc5,Idc6,Idc7,Idc8,Q,V1,V2,V3
0,01/01/2025 0:00,"4,69333342",0,0,"223,4346189","5,776875518","13,12315993",0,"8,329722649",0,...,0,0,0,0,0,0,0,"381,7256941","382,0004276","381,5076763"
1,01/01/2025 0:15,"4,778227202",0,0,"230,5402257","6,300436193",0,0,"8,129726934",0,...,0,0,0,0,0,0,0,"382,64233","382,7545673","382,0091918"
2,01/01/2025 0:30,"4,701774968",0,0,"236,7170294","6,500882991",0,0,"7,92973122",0,...,0,0,0,0,0,0,0,"379,7375055","380,1679837","379,2757975"
3,01/01/2025 0:45,"4,641534331",0,0,"237,0765359","6,044294316",0,0,"7,734364006",0,...,0,0,0,0,0,0,0,"378,2676107","379,5482362","378,593559"
4,01/01/2025 1:00,"4,613333581",0,0,"236,6212896","5,554435369",0,0,"7,593589895",0,...,0,0,0,0,0,0,0,"377,6498307","379,3398828","378,372256"


# 2. Análisis exploratorio

Sustituición de "," por "." para el tratamiento de los datos.

In [3]:
df_test = df_test.apply(lambda x: x.str.replace(",", ".", regex=True) if x.dtype == "object" else x)
df_test.head()

,Fecha_Hora,Temp_amb,Radiacion_1,Radiacion_2,Dir_viento,Vel_viento,Energia,Potencia,Oil_Temp,cosPhi,...,Idc3,Idc4,Idc5,Idc6,Idc7,Idc8,Q,V1,V2,V3
0,01/01/2025 0:00,4.69333342,0,0,223.4346189,5.776875518,13.12315993,0,8.329722649,0,...,0,0,0,0,0,0,0,381.7256941,382.0004276,381.5076763
1,01/01/2025 0:15,4.778227202,0,0,230.5402257,6.300436193,0,0,8.129726934,0,...,0,0,0,0,0,0,0,382.64233,382.7545673,382.0091918
2,01/01/2025 0:30,4.701774968,0,0,236.7170294,6.500882991,0,0,7.92973122,0,...,0,0,0,0,0,0,0,379.7375055,380.1679837,379.2757975
3,01/01/2025 0:45,4.641534331,0,0,237.0765359,6.044294316,0,0,7.734364006,0,...,0,0,0,0,0,0,0,378.2676107,379.5482362,378.593559
4,01/01/2025 1:00,4.613333581,0,0,236.6212896,5.554435369,0,0,7.593589895,0,...,0,0,0,0,0,0,0,377.6498307,379.3398828,378.372256


Se transforma la serie temporal en una regresión, para ello se extraen las características más utiles de la columna Fecha_Hora y se generan nuevas columnas que nos ayuden al tratamiento de los datos.

In [4]:
# Convertimos la columna a tipo datetime 
df_test["Fecha_Hora"] = pd.to_datetime(df_test["Fecha_Hora"], format="%d/%m/%Y %H:%M", dayfirst=True)


# 📌 Desglose de la fecha en variables útiles
df_test["Año"] = df_test["Fecha_Hora"].dt.year
df_test["Mes"] = df_test["Fecha_Hora"].dt.month
df_test["Día"] = df_test["Fecha_Hora"].dt.day
df_test["Hora"] = df_test["Fecha_Hora"].dt.hour
df_test["Minuto"] = df_test["Fecha_Hora"].dt.minute


# 📌 Función para asignar la estación del año
def get_estacion(mes, dia):
    if (mes == 12 and dia >= 21) or (mes <= 3 and (mes != 3 or dia < 20)):
        return "Invierno"
    elif (mes == 3 and dia >= 20) or (mes < 6) or (mes == 6 and dia < 21):
        return "Primavera"
    elif (mes == 6 and dia >= 21) or (mes < 9) or (mes == 9 and dia < 23):
        return "Verano"
    else:
        return "Otoño"

# Aplicamos la función a cada fila
df_test["Estacion"] = df_test.apply(lambda x: get_estacion(x["Mes"], x["Día"]), axis=1)

df_test.drop(columns=["Fecha_Hora"], inplace=True)


df_test.head()


,Temp_amb,Radiacion_1,Radiacion_2,Dir_viento,Vel_viento,Energia,Potencia,Oil_Temp,cosPhi,Frecuencia,...,Q,V1,V2,V3,Año,Mes,Día,Hora,Minuto,Estacion
0,4.69333342,0,0,223.4346189,5.776875518,13.12315993,0,8.329722649,0,50,...,0,381.7256941,382.0004276,381.5076763,2025,1,1,0,0,Invierno
1,4.778227202,0,0,230.5402257,6.300436193,0,0,8.129726934,0,50,...,0,382.64233,382.7545673,382.0091918,2025,1,1,0,15,Invierno
2,4.701774968,0,0,236.7170294,6.500882991,0,0,7.92973122,0,49.99984573,...,0,379.7375055,380.1679837,379.2757975,2025,1,1,0,30,Invierno
3,4.641534331,0,0,237.0765359,6.044294316,0,0,7.734364006,0,49.99787208,...,0,378.2676107,379.5482362,378.593559,2025,1,1,0,45,Invierno
4,4.613333581,0,0,236.6212896,5.554435369,0,0,7.593589895,0,49.99537259,...,0,377.6498307,379.3398828,378.372256,2025,1,1,1,0,Invierno


Se generan las siguientes columnas a traves de la variable Fecha_Hora
- Año.
- Mes.
- Día.
- Día de la semana (Lunes=0 ; Domingo=6)
- Hora.
- Minuto, en este caso (0,15,30,45).
- Estación.

In [5]:
# Tamaño del dataset (filas, columnas)
print("Dimensiones del dataset:", df_test.shape)

Dimensiones del dataset: (5665, 31)


In [6]:
# Información general del dataframe
print(df_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5665 entries, 0 to 5664
Data columns (total 31 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Temp_amb       5665 non-null   object
 1   Radiacion_1    5665 non-null   object
 2   Radiacion_2    5665 non-null   object
 3   Dir_viento     5665 non-null   object
 4   Vel_viento     5665 non-null   object
 5   Energia        5665 non-null   object
 6   Potencia       5665 non-null   object
 7   Oil_Temp       5665 non-null   object
 8   cosPhi         5665 non-null   object
 9   Frecuencia     5665 non-null   object
 10  HeatsinkRTemp  5665 non-null   object
 11  HeatsinkSTemp  5665 non-null   object
 12  HeatsinkTTemp  5665 non-null   object
 13  Idc1           5665 non-null   object
 14  Idc2           5665 non-null   object
 15  Idc3           5665 non-null   object
 16  Idc4           5665 non-null   object
 17  Idc5           5665 non-null   object
 18  Idc6           5665 non-null

El dataset consta de 33 columnas y 105216 filas, con ausencia de nulos y huecos, todas las columnas son del tipo object , se va realizar una conversión de las columnas tipo object a numéricas para realizar el análisis de las mismas.

In [7]:
cols_a_convertir = ["Radiacion_1","Radiacion_2","Dir_viento","Vel_viento","Energia","Potencia","Oil_Temp","cosPhi","Frecuencia","HeatsinkRTemp","HeatsinkSTemp",
                    "HeatsinkTTemp","Idc1","Idc2","Idc3","Idc4","Idc5","Idc6","Idc7","Idc8","Temp_amb","Q","V1","V2","V3","Año","Mes","Día","Hora","Minuto"]
df_test[cols_a_convertir] = df_test[cols_a_convertir].apply(pd.to_numeric, errors='coerce')


In [8]:
# Descripción estadística
df_test.describe()

,Temp_amb,Radiacion_1,Radiacion_2,Dir_viento,Vel_viento,Energia,Potencia,Oil_Temp,cosPhi,Frecuencia,...,Idc8,Q,V1,V2,V3,Año,Mes,Día,Hora,Minuto
count,5665.000000,5665.000000,5665.000000,5665.000000,5665.000000,5665.000000,5665.000000,5665.000000,5665.000000,5665.000000,...,5665.000000,5665.000000,5665.000000,5665.000000,5665.000000,5665.0,5665.000000,5665.000000,5665.000000,5665.000000
mean,3.577209,63.638076,62.745382,164.765210,3.289273,1522.159318,135.322171,19.046485,0.351936,49.033635,...,17.066796,5.189356,374.305560,374.649744,374.773252,2025.0,1.474846,15.285613,11.497970,22.496028
std,4.216273,139.512540,136.792766,80.123332,2.128499,2289.157080,301.186146,12.746270,0.463241,6.868562,...,37.292175,19.605963,52.504115,52.585113,52.576566,0.0,0.499764,8.578874,6.923873,16.773174
min,-5.492936,0.000000,0.000000,0.000000,0.000000,0.000000,-3.721746,-0.176281,-0.861716,0.000000,...,0.000000,-3.629260,0.000000,0.000000,0.000000,2025.0,1.000000,1.000000,0.000000,0.000000
25%,0.200000,0.000000,0.000000,109.433276,1.754769,0.000000,0.000000,8.563199,0.000000,49.982247,...,0.000000,0.000000,379.699849,379.582841,380.048725,2025.0,1.000000,8.000000,5.000000,0.000000
50%,3.100427,0.000000,0.000000,152.261153,2.774236,457.392070,0.000000,16.873371,0.000000,49.995545,...,0.000000,0.000000,381.264039,382.169019,381.825525,2025.0,1.000000,15.000000,11.000000,15.000000
75%,6.580500,51.025843,51.221692,226.154711,4.527034,2015.726133,107.497840,26.799592,0.998320,50.007751,...,15.114632,0.000000,382.952954,384.241736,383.630160,2025.0,2.000000,23.000000,17.000000,30.000000
max,17.225289,779.412229,772.419604,349.438632,15.964609,9881.950195,1633.718598,65.753198,0.999937,50.058353,...,208.667679,154.975360,394.708415,392.525847,394.492370,2025.0,3.000000,31.000000,23.000000,45.000000


# 3. Feature Engineer

Una vez realizada la matriz de correlación se observa que hay varias columnas que nos dan una información similar y se pueden unir entre ellas para reducir el tamaño del
dataframe y poder tratar de una manera más optima los datos:
  - Se genera una nueva columna llamada Rad_prom que resulta de realizar el promedio de la Radiacion_1 y Radiacion_2.
  - Se eliminan las columnas Radiacion_1 y Radiacion_2.
  - Se genera una nueva columna llamada HeatsinkTemp_prom que resulta de realizar el promedio de la HeatsinkR, HeatsinkS, HeatsinkT.
  - Se eliminan las columnas HeatsinkR, HeatsinkS, HeatsinkT.
  - Se genera una nueva columna llamada Idc_total que resulta de realizar el sumatorio de todas las Idc que componen el dataframe.
  - Se eliminan las columnas Idc1, Idc2, Idc3, Idc4, Idc5, Idc6, Idc7, Idc8.
  - Se genera una nueva columna llamada V_total que resulta de realizar el sumatorio de V1, V2 y V3.
  - Se eliminan las columnas V1, V2, V3.

In [9]:
df_test['Rad_prom'] = df_test[['Radiacion_1', 'Radiacion_2']].mean(axis=1)
df_test = df_test.drop(columns=['Radiacion_1', 'Radiacion_2'])

df_test['HeatsinkTemp_prom'] = df_test[['HeatsinkRTemp', 'HeatsinkSTemp', 'HeatsinkTTemp']].mean(axis=1)
df_test = df_test.drop(columns=['HeatsinkRTemp', 'HeatsinkSTemp', 'HeatsinkTTemp'])

df_test['Idc_total'] = df_test[['Idc1', 'Idc2', 'Idc3', 'Idc4', 'Idc5', 'Idc6', 'Idc7', 'Idc8']].sum(axis=1)
df_test = df_test.drop(columns=['Idc1', 'Idc2', 'Idc3', 'Idc4', 'Idc5', 'Idc6', 'Idc7', 'Idc8'])

df_test['V_total'] = df_test[['V1', 'V2', 'V3']].sum(axis=1)
df_test = df_test.drop(columns=['V1', 'V2', 'V3'])

df_test.head()

,Temp_amb,Dir_viento,Vel_viento,Energia,Potencia,Oil_Temp,cosPhi,Frecuencia,Q,Año,Mes,Día,Hora,Minuto,Estacion,Rad_prom,HeatsinkTemp_prom,Idc_total,V_total
0,4.693333,223.434619,5.776876,13.12316,0.0,8.329723,0.0,50.000000,0.0,2025,1,1,0,0,Invierno,0.0,7.165451,0.0,1145.233798
1,4.778227,230.540226,6.300436,0.00000,0.0,8.129727,0.0,50.000000,0.0,2025,1,1,0,15,Invierno,0.0,7.166667,0.0,1147.406089
2,4.701775,236.717029,6.500883,0.00000,0.0,7.929731,0.0,49.999846,0.0,2025,1,1,0,30,Invierno,0.0,7.166668,0.0,1139.181287
3,4.641534,237.076536,6.044294,0.00000,0.0,7.734364,0.0,49.997872,0.0,2025,1,1,0,45,Invierno,0.0,7.174754,0.0,1136.409406
4,4.613334,236.621290,5.554435,0.00000,0.0,7.593590,0.0,49.995373,0.0,2025,1,1,1,0,Invierno,0.0,7.193001,0.0,1135.361969


### Transformaciones de las variables generadas de la columna Fecha_Hora

1. Se necesita transformar la columna estacion a numerica para incluirla en el modelo.

In [10]:
# Crear un diccionario de mapeo
estacion_map = {
    "Invierno": 1,
    "Primavera": 2,
    "Verano": 3,
    "Otoño": 4
}

# Reemplazar valores de "Estacion" con números
df_test["Estacion"] = df_test["Estacion"].map(estacion_map)

2. A la columnas Minuto, Hora, Dia y Mes se les realiza una transformación con la función seno, esto nos permite representar estos valores de una forma que mantiene su naturaleza cíclica.

In [11]:
# Convertir los minutos a valores cíclicos usando seno y coseno
df_test["Minuto_sin"] = np.sin(2 * np.pi * df_test["Minuto"] / 60)  # Usamos el seno

# Hora: de 0 a 23 (24 horas)
df_test["Hora_sin"] = np.sin(2 * np.pi * df_test["Hora"] / 24)

# Día: de 0 a 6 (7 días de la semana)
df_test["Dia_sin"] = np.sin(2 * np.pi * df_test["Día"] / 7)

# Mes: de 1 a 12 (12 meses del año)
df_test["Mes_sin"] = np.sin(2 * np.pi * df_test["Mes"] / 12)

df_test.head()

,Temp_amb,Dir_viento,Vel_viento,Energia,Potencia,Oil_Temp,cosPhi,Frecuencia,Q,Año,...,Minuto,Estacion,Rad_prom,HeatsinkTemp_prom,Idc_total,V_total,Minuto_sin,Hora_sin,Dia_sin,Mes_sin
0,4.693333,223.434619,5.776876,13.12316,0.0,8.329723,0.0,50.000000,0.0,2025,...,0,1,0.0,7.165451,0.0,1145.233798,0.000000e+00,0.000000,0.781831,0.5
1,4.778227,230.540226,6.300436,0.00000,0.0,8.129727,0.0,50.000000,0.0,2025,...,15,1,0.0,7.166667,0.0,1147.406089,1.000000e+00,0.000000,0.781831,0.5
2,4.701775,236.717029,6.500883,0.00000,0.0,7.929731,0.0,49.999846,0.0,2025,...,30,1,0.0,7.166668,0.0,1139.181287,5.665539e-16,0.000000,0.781831,0.5
3,4.641534,237.076536,6.044294,0.00000,0.0,7.734364,0.0,49.997872,0.0,2025,...,45,1,0.0,7.174754,0.0,1136.409406,-1.000000e+00,0.000000,0.781831,0.5
4,4.613334,236.621290,5.554435,0.00000,0.0,7.593590,0.0,49.995373,0.0,2025,...,0,1,0.0,7.193001,0.0,1135.361969,0.000000e+00,0.258819,0.781831,0.5


In [12]:
df_test = df_test.drop(columns=['Minuto','Mes','Día','Hora'])


In [13]:
df_test.to_csv('C:/Users/javier.sdiaz/Proyecto ML_2025/data/processed/Test_Temp_int_proc.csv', index=False, encoding='utf-8')
